# Step 2: EuroSpeech dataset and Granite 4 baseline (EN/DE)

This compact notebook prepares the approved EuroSpeech artifact and optionally
runs a no-context Granite 4 baseline. It borrows only a simple staged layout;
all reusable work stays in `src/ner_asr/`.

* Dataset: EuroSpeech `uk` (English) and `germany` (German), official test split.
* Model: `ibm-granite/granite-4.0-1b-speech`, always called with `prompt=None` for the baseline.
* Step 2 artifact: one parquet with exactly 1,000 English and 1,000 German rows.
* Out of scope: NER, keyword-conditioned runs, error mining, and fine-tuning.
* Nothing downloads by default. Each networked action has its own explicit switch.

## 1. Configuration

Choose a language config. The default is English; rerun the notebook with
`NER_ASR_CONFIG=config/eurospeech_de.yaml` for the 250-record German pilot.
Set a switch to `1` only when you intend the action:
`NER_ASR_RUN_SELECTION=1` streams the pilot split,
`NER_ASR_RUN_INFERENCE=1` downloads/runs Granite, and
`NER_ASR_EXPORT_DELIVERABLE=1` creates the parquet artifact.

In [ ]:
import json
import os
import sys
from pathlib import Path

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO / "src"))

from ner_asr import pipeline
from ner_asr.config import load_config
from ner_asr.selection import read_manifest

CONFIG = os.environ.get("NER_ASR_CONFIG", "config/eurospeech_en.yaml")
DELIVERABLE_DE_CONFIG = os.environ.get("NER_ASR_CONFIG_DE", "config/eurospeech_de.yaml")
MODE = os.environ.get("NER_ASR_MODE", "pilot")
RUN_INFERENCE = os.environ.get("NER_ASR_RUN_INFERENCE", "0") == "1"
EXPORT_STEP2_DELIVERABLE = os.environ.get("NER_ASR_EXPORT_DELIVERABLE", "0") == "1"

cfg = load_config(CONFIG, mode=MODE)
if MODE == "full" and not cfg.run_name.endswith("_full"):
    cfg.run_name += "_full"
IS_FIXTURE = cfg.dataset.adapter == "fixture"
RUN_SELECTION = IS_FIXTURE or os.environ.get("NER_ASR_RUN_SELECTION", "0") == "1"

print(json.dumps({
    "config": CONFIG,
    "deliverable_de_config": DELIVERABLE_DE_CONFIG,
    "dataset": cfg.dataset.name,
    "split": cfg.dataset.split,
    "model": cfg.model.name,
    "run_selection": RUN_SELECTION,
    "run_inference": RUN_INFERENCE,
    "export_deliverable": EXPORT_STEP2_DELIVERABLE,
}, indent=2))

#to run in the jupyterhub 
os.environ["NER_ASR_RUN_SELECTION"] = "1"
os.environ["NER_ASR_RUN_INFERENCE"] = "1"
os.environ["NER_ASR_EXPORT_DELIVERABLE"] = "1"

## 2. What one dataset row looks like

This is a safe structural preview of one final parquet row. It uses a readable
placeholder instead of real binary audio and does not open or download data.

In [ ]:
from ner_asr.previews import example_deliverable_row

print(json.dumps(example_deliverable_row(
    language=cfg.dataset.language,
    source_split=cfg.dataset.split,
), indent=2))

## 3. What Granite receives and returns

This shows the exact no-context request shape used by the baseline. It is an
example only: no model weights are loaded and no inference is run.

In [ ]:
from ner_asr.previews import granite_baseline_example

print(json.dumps(granite_baseline_example(model_name=cfg.model.name), indent=2))

## 4. Select or reuse a fixed split

Selection is deterministic and never mixes official splits. If no manifest is
available and selection is disabled, this cell remains local and simply reports
what needs to be enabled.

In [ ]:
if IS_FIXTURE:
    import subprocess
    subprocess.run([sys.executable, str(REPO / "tests" / "make_fixture.py")], check=True)

if RUN_SELECTION:
    records, report = pipeline.step_select(cfg)
elif cfg.manifest_path.exists():
    records, report = read_manifest(cfg.manifest_path), None
    print("Using existing manifest:", cfg.manifest_path)
else:
    records, report = [], None
    print("Selection skipped. Set NER_ASR_RUN_SELECTION=1 to stream the selected split.")

if records:
    hours = sum(record.duration_seconds or 0 for record in records) / 3600
    first = records[0]
    print(f"{len(records)} recordings | {hours:.3f} h | first: {first.sample_id}")
    try:
        from IPython.display import Audio, display
        from ner_asr.audio import load_audio
        display(Audio(load_audio(first.audio_ref), rate=16_000))
    except Exception as exc:
        print("audio preview unavailable:", exc)

## 5. Granite baseline

The baseline uses audio only—no reference text, entity list, or keyword prompt
reaches the model. Existing predictions can be evaluated without loading the
model.

In [ ]:
transcriber = None
if IS_FIXTURE:
    from types import SimpleNamespace

    class FixtureTranscriber:
        def transcribe_batch(self, paths, prompt=None):
            assert prompt is None
            return [SimpleNamespace(
                prediction="fixture output", raw_output="fixture output",
                n_generated_tokens=2, hit_token_limit=False, runtime_s=0.0,
            ) for _ in paths]

    transcriber = FixtureTranscriber()
elif RUN_INFERENCE and records:
    from ner_asr.transcribe import Granite4ASRTranscriber
    transcriber = Granite4ASRTranscriber(cfg.model)
    print("Loaded", cfg.model.name, "on", transcriber.device)
elif RUN_INFERENCE:
    print("Inference requested, but no manifest is available. Run selection or reuse a manifest first.")
else:
    print("Inference skipped. Set NER_ASR_RUN_INFERENCE=1 to load Granite.")

if transcriber is not None and records:
    pipeline.step_transcribe(cfg, transcriber=transcriber)

summary = pipeline.step_evaluate(cfg) if cfg.predictions_path.exists() else None
if summary:
    print("Mean WER:", summary["per_recording_wer_mean"])
    print("Mean CER:", summary["per_recording_cer_mean"])
else:
    print("No predictions to evaluate yet.")

## 6. Step 2 parquet artifact 

This is separate from model inference. It streams about 1,200 candidates for
each language from the official test split, writes one embedded-audio parquet
with exactly 1,000 English and 1,000 German valid 2–20 second rows, reloads it,
and displays one WAV sample. The final file is
`data/deliverable/step2_eurospeech_en_de.parquet`; every row carries its
`language` value.

In [ ]:
if EXPORT_STEP2_DELIVERABLE:
    de_cfg = load_config(DELIVERABLE_DE_CONFIG, mode=MODE)
    artifact, report = pipeline.step_export_deliverable(cfg, de_cfg)
    print(json.dumps(report.to_dict(), indent=2))
    try:
        from IPython.display import Audio, display
        from ner_asr.deliverable import sample_audio_bytes
        display(Audio(data=sample_audio_bytes(artifact)))
    except Exception as exc:
        print("parquet audio preview unavailable:", exc)
else:
    print("Parquet export skipped. Set NER_ASR_EXPORT_DELIVERABLE=1 to run it.")

In [ ]:
#to check the parquet file
from pathlib import Path
import pandas as pd

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
parquet_path = REPO / "data" / "deliverable" / "step2_eurospeech_en_de.parquet"

if not parquet_path.exists():
    found = list(REPO.rglob("*.parquet"))
    print("File not found at:", parquet_path)
    print("Parquet files currently in repo:", found)
    print("\n-> Make sure you ran Section 6 with EXPORT_STEP2_DELIVERABLE = True first!")
else:
    df = pd.read_parquet(parquet_path)
    print("Loaded:", parquet_path)
    print("Total rows:", len(df))
    print("Columns:", df.columns.tolist())
    print("\nRows per language:\n", df["language"].value_counts())

    cols_no_audio = [c for c in df.columns if c != "audio"]
    pd.set_option("display.max_colwidth", None)
    display(df[cols_no_audio].head(10))